In [1]:
import numpy as np
np.random.seed(0)
import matplotlib.pyplot as plt
import torch
from utils.make_manifold_data import make_manifold_data
from utils.activation_extractor import extractor
from analyze_pytorch import analyze
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
from torch.utils.data import TensorDataset
from BIMT_2D import BioMLP2D
from scipy.linalg import qr
from collections import defaultdict
import os
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

2023-06-06 15:08:01.385522: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from manifold_analysis_correlation import manifold_analysis_corr


In [3]:
seed = 1
np.random.seed(seed)
torch.manual_seed(seed)

In [4]:
sampled_per_class = 2
examples_per_class = 50

d_in = 784
d_out = 10

# Load MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='../../FisherInformation/data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = datasets.MNIST(root='../../FisherInformation/data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [5]:
with torch.no_grad():
    data = make_manifold_data(train_dataset, sampled_per_class, examples_per_class, seed=0)

In [7]:
model = BioMLP2D(shp=[784, 100, 100, 10])

activations = extractor(model, data, layer_nums=[1,2,3])
print(activations.keys())

odict_keys(['layer_0_Input', 'layer_1_Linear', 'layer_2_Linear', 'layer_3_Linear'])


In [8]:

res_dict = defaultdict(lambda: defaultdict(dict))
for indx, model_name in enumerate(os.listdir('models/bimt_cnn')):
    model.load_state_dict(torch.load('models/bimt_cnn/{}'.format(model_name)))
    activations = extractor(model, data, layer_nums=[1,2,3])
    radii, dimensions, capacities, r0s, Ks = [], [], [], [], []
    for k, X, in activations.items():
        
        a, r, d, r0, K = manifold_analysis_corr(X, 0, 300, n_reps=1)
        radii.append(r)
        dimensions.append(d)
        capacities.append(a)
        r0s.append(r0)
        Ks.append(K)

    print(radii, dimensions, capacities, r0s, Ks)
    

ValueError: too many values to unpack (expected 2)

In [136]:
res_dict = dict(sorted(res_dict.items()))
    

In [137]:
print(res_dict.keys())

dict_keys([0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400, 2500, 2600, 2700, 2800, 2900, 3000, 3100, 3200, 3300, 3400, 3500, 3600, 3700, 3800, 3900, 4000, 4100, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 4900, 5000, 5100, 5200, 5300, 5400, 5500, 5600, 5700, 5800, 5900, 6000, 6100, 6200, 6300, 6400, 6500, 6600, 6700, 6800, 6900, 7000, 7100, 7200, 7300, 7400, 7500, 7600, 7700, 7800, 7900, 8000, 8100, 8200, 8300, 8400, 8500, 8600, 8700, 8800, 8900, 9000, 9100, 9200, 9300, 9400, 9500, 9600, 9700, 9800, 9900, 10000])


In [141]:
final_results = defaultdict(lambda: defaultdict(dict))
for indx, key in enumerate(res_dict.keys()):
    if indx == 0:
        final_results["Input Layer"]["Radius"][0] = [res_dict[key]["radius_0"][0]]
        final_results["Input Layer"]["Radius"][1] = [res_dict[key]["radius_1"][0]]
    
        final_results["Input Layer"]["Dimension"][0] = [res_dict[key]["dimension_0"][0]]
        final_results["Input Layer"]["Dimension"][1] = [res_dict[key]["dimension_1"][0]]
    
        final_results["Input Layer"]["Capacity"] = [res_dict[key]["capacity"][0]]
    
        final_results["Hidden Layer 1"]["Radius"][0] = [res_dict[key]["radius_0"][1]]
        final_results["Hidden Layer 1"]["Radius"][1] = [res_dict[key]["radius_1"][1]]
    
        final_results["Hidden Layer 1"]["Dimension"][0] = [res_dict[key]["dimension_0"][1]]
        final_results["Hidden Layer 1"]["Dimension"][1] = [res_dict[key]["dimension_1"][1]]
    
        final_results["Hidden Layer 1"]["Capacity"] = [res_dict[key]["capacity"][1]]
    
        final_results["Hidden Layer 2"]["Radius"][0] = [res_dict[key]["radius_0"][2]]
        final_results["Hidden Layer 2"]["Radius"][1] = [res_dict[key]["radius_1"][2]]
    
        final_results["Hidden Layer 2"]["Dimension"][0] = [res_dict[key]["dimension_0"][2]]
        final_results["Hidden Layer 2"]["Dimension"][1] = [res_dict[key]["dimension_1"][2]]
    
        final_results["Hidden Layer 2"]["Capacity"] = [res_dict[key]["capacity"][2]]
    
        final_results["Output Layer"]["Radius"][0] = [res_dict[key]["radius_0"][3]]
        final_results["Output Layer"]["Radius"][1] = [res_dict[key]["radius_1"][3]]
    
        final_results["Output Layer"]["Dimension"][0] = [res_dict[key]["dimension_0"][3]]
        final_results["Output Layer"]["Dimension"][1] = [res_dict[key]["dimension_1"][3]]
    
        final_results["Output Layer"]["Capacity"] = [res_dict[key]["capacity"][3]]

    else:
        final_results["Input Layer"]["Radius"][0].append(res_dict[key]["radius_0"][0])
        final_results["Input Layer"]["Radius"][1].append(res_dict[key]["radius_1"][0])

        final_results["Input Layer"]["Dimension"][0].append(res_dict[key]["dimension_0"][0])
        final_results["Input Layer"]["Dimension"][1].append(res_dict[key]["dimension_1"][0])

        final_results["Input Layer"]["Capacity"].append(res_dict[key]["capacity"][0])

        final_results["Hidden Layer 1"]["Radius"][0].append(res_dict[key]["radius_0"][1])
        final_results["Hidden Layer 1"]["Radius"][1].append(res_dict[key]["radius_1"][1])

        final_results["Hidden Layer 1"]["Dimension"][0].append(res_dict[key]["dimension_0"][1])
        final_results["Hidden Layer 1"]["Dimension"][1].append(res_dict[key]["dimension_1"][1])

        final_results["Hidden Layer 1"]["Capacity"].append(res_dict[key]["capacity"][1])

        final_results["Hidden Layer 2"]["Radius"][0].append(res_dict[key]["radius_0"][2])
        final_results["Hidden Layer 2"]["Radius"][1].append(res_dict[key]["radius_1"][2])

        final_results["Hidden Layer 2"]["Dimension"][0].append(res_dict[key]["dimension_0"][2])
        final_results["Hidden Layer 2"]["Dimension"][1].append(res_dict[key]["dimension_1"][2])

        final_results["Hidden Layer 2"]["Capacity"].append(res_dict[key]["capacity"][2])

        final_results["Output Layer"]["Radius"][0].append(res_dict[key]["radius_0"][3])
        final_results["Output Layer"]["Radius"][1].append(res_dict[key]["radius_1"][3])

        final_results["Output Layer"]["Dimension"][0].append(res_dict[key]["dimension_0"][3])
        final_results["Output Layer"]["Dimension"][1].append(res_dict[key]["dimension_1"][3])

        final_results["Output Layer"]["Capacity"].append(res_dict[key]["capacity"][3])



In [145]:
print(final_results["Input Layer"]["Capacity"])

[-20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.303560256958008, -20.30356025